# Modulo 6 · Lezione 7 — Progetto: Analisi Esplorativa dei Dati (EDA)

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

```
Un EDA completo ha 5 fasi:

  1. CARICARE  → importare i dati, capire la struttura
  2. ESPLORARE → dimensioni, tipi, statistiche iniziali
  3. PULIRE    → valori mancanti, outlier, incoerenze
  4. ANALIZZARE → distribuzioni, correlazioni, pattern
  5. CONCLUDERE → domande di business a cui rispondere, ipotesi per la modellazione

Dataset: Titanic (passeggeri della nave da crociera)
Domanda di business: quali fattori determinano la sopravvivenza?
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

np.random.seed(42)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 110)
print("Librerie pronte")

---
## Fase 1 — Caricamento dei dati

Usiamo il dataset Titanic, disponibile tramite seaborn senza installazioni aggiuntive.

In [ ]:
import seaborn as sns

# Dataset classico: 891 passeggeri del Titanic
df_raw = sns.load_dataset('titanic')

print("=== Dataset caricato ===")
print(f"Dimensioni: {df_raw.shape}")
print(f"\nColonne:")
for col in df_raw.columns:
    print(f"  {col:20} {df_raw[col].dtype}")

print("\nPrime righe:")
print(df_raw.head())

print("\nDescrizione delle colonne:")
descripcion = {
    'survived':  'Sopravvissuto (1=Sì, 0=No)',
    'pclass':    'Classe del biglietto (1=Prima, 2=Seconda, 3=Terza)',
    'sex':       'Sesso del passeggero',
    'age':       'Età in anni',
    'sibsp':     'N° di fratelli/coniuge a bordo',
    'parch':     'N° di genitori/figli a bordo',
    'fare':      'Tariffa del biglietto (£)',
    'embarked':  'Porto di imbarco (C=Cherbourg, Q=Queenstown, S=Southampton)',
    'class':     'Classe (testo)',
    'who':       'Categoria (man/woman/child)',
    'adult_male':'Uomo adulto (booleano)',
    'deck':      'Ponte della cabina',
    'embark_town':'Città di imbarco',
    'alive':     'Sopravvissuto (yes/no)',
    'alone':     'Viaggiava da solo',
}
for col, desc in descripcion.items():
    print(f"  {col:15}: {desc}")

---
## Fase 2 — Esplorazione iniziale

In [ ]:
df = df_raw.copy()

print("=== Statistiche descrittive ===")
print(df.describe(include='all').round(2))

print("\n=== Valori mancanti ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct_%': missing_pct})
print(missing_df[missing_df['count'] > 0].sort_values('count', ascending=False))

print("\n=== Variabile obiettivo ===")
print(df['survived'].value_counts())
print(f"Tasso di sopravvivenza: {df['survived'].mean()*100:.1f}%")

print("\n=== Distribuzione delle classi ===")
print(df['pclass'].value_counts().sort_index())

print("\n=== Distribuzione per sesso ===")
print(df['sex'].value_counts())

In [ ]:
# Pannello di esplorazione iniziale
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 4, figure=fig)

# Sopravvivenza
ax0 = fig.add_subplot(gs[0, 0])
conteos = df['survived'].value_counts()
ax0.bar(['Non sopravvissuto', 'Sopravvissuto'], conteos.values, color=['#e74c3c', '#2ecc71'])
ax0.set_title('Sopravvivenza')
for i, v in enumerate(conteos.values):
    ax0.text(i, v + 5, str(v), ha='center')

# Per classe
ax1 = fig.add_subplot(gs[0, 1])
df.groupby('pclass')['survived'].mean().plot(kind='bar', ax=ax1, color='steelblue')
ax1.set_title('Sopravvivenza per classe')
ax1.set_ylabel('Tasso')
ax1.set_xticklabels(['1ª', '2ª', '3ª'], rotation=0)

# Per sesso
ax2 = fig.add_subplot(gs[0, 2])
df.groupby('sex')['survived'].mean().plot(kind='bar', ax=ax2, color=['#9b59b6', '#e67e22'])
ax2.set_title('Sopravvivenza per sesso')
ax2.set_ylabel('Tasso')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)

# Porto di imbarco
ax3 = fig.add_subplot(gs[0, 3])
df.groupby('embarked')['survived'].mean().plot(kind='bar', ax=ax3, color='teal')
ax3.set_title('Sopravvivenza per imbarco')
ax3.set_ylabel('Tasso')
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)

# Distribuzione dell'età
ax4 = fig.add_subplot(gs[1, 0:2])
df[df['survived']==0]['age'].dropna().hist(bins=30, alpha=0.6, color='#e74c3c', label='Non sopravvissuto', ax=ax4)
df[df['survived']==1]['age'].dropna().hist(bins=30, alpha=0.6, color='#2ecc71', label='Sopravvissuto', ax=ax4)
ax4.set_title("Distribuzione dell'età per sopravvivenza")
ax4.set_xlabel('Età'); ax4.legend()

# Distribuzione della tariffa
ax5 = fig.add_subplot(gs[1, 2:4])
df[df['survived']==0]['fare'].hist(bins=40, alpha=0.6, color='#e74c3c', label='Non sopravvissuto', ax=ax5)
df[df['survived']==1]['fare'].hist(bins=40, alpha=0.6, color='#2ecc71', label='Sopravvissuto', ax=ax5)
ax5.set_title('Distribuzione della tariffa per sopravvivenza')
ax5.set_xlabel('Tariffa (£)'); ax5.legend()
ax5.set_xlim(0, 300)

plt.suptitle('Analisi Esplorativa Titanic — Visione Generale', fontsize=13)
plt.tight_layout(); plt.show()

---
## Fase 3 — Pulizia dei dati

In [ ]:
print("=== Piano di pulizia ===")
print("age      → 177 mancanti (20%) → imputare con la mediana per classe+sesso")
print("embarked → 2 mancanti → imputare con la moda")
print("deck     → 688 mancanti (77%) → troppi, creare categoria 'Sconosciuto'")
print("fare     → nessun outlier estremo ma da verificare")
print("Colonne ridondanti: class, alive, embark_town → eliminare (duplicano informazioni)")

# Imputare l'età con la mediana per classe e sesso
mediana_edad = df.groupby(['pclass', 'sex'])['age'].median()
print("\nMediana dell'età per classe e sesso:")
print(mediana_edad)

df['age'] = df.groupby(['pclass', 'sex'])['age'].transform(
    lambda x: x.fillna(x.median())
)

# Imputare embarked con la moda
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# Ponte: sconosciuto per i mancanti
df['deck'] = df['deck'].astype(str).replace('nan', 'Sconosciuto')

# Eliminare colonne ridondanti
df = df.drop(columns=['class', 'alive', 'embark_town', 'who', 'adult_male'])

# Verificare
print("\n=== Valori mancanti dopo la pulizia ===")
print(df.isnull().sum())
print(f"\nDataset pulito: {df.shape}")

# Outlier nella tariffa
Q1, Q3 = df['fare'].quantile([0.25, 0.75])
IQR    = Q3 - Q1
outliers_fare = df[df['fare'] > Q3 + 3*IQR]
print(f"\nOutlier estremi nella tariffa (>3×IQR): {len(outliers_fare)}")
print(f"Tariffa massima: {df['fare'].max():.2f}£  (percentile 99: {df['fare'].quantile(0.99):.2f}£)")
print("→ Manteniamo gli outlier: rappresentano cabine di lusso, contengono informazione")

---
## Fase 4 — Analisi approfondita

In [ ]:
print("=== ANALISI 1: Tasso di sopravvivenza per variabili chiave ===")

# Creare gruppi di età
df['grupo_edad'] = pd.cut(df['age'], bins=[0, 12, 18, 35, 60, 100],
                          labels=['Bambino', 'Adolescente', 'Giovane adulto', 'Adulto', 'Anziano'])

# Creare gruppi di tariffa
df['grupo_tarifa'] = pd.qcut(df['fare'], q=4,
                             labels=['Economico', 'Normale', 'Superiore', 'Lusso'])

# Tabella di sopravvivenza per variabili
vars_analizar = ['pclass', 'sex', 'embarked', 'alone', 'grupo_edad']
for var in vars_analizar:
    tabla = df.groupby(var)['survived'].agg(['mean','count'])
    tabla.columns = ['tasa_supervivencia', 'n']
    tabla['tasa_supervivencia'] = (tabla['tasa_supervivencia'] * 100).round(1)
    print(f"\nSopravvivenza per {var}:")
    print(tabla.sort_values('tasa_supervivencia', ascending=False).to_string())

In [ ]:
print("=== ANALISI 2: Interazioni tra variabili ===")

# Sesso × Classe
tabla_cruzada = df.groupby(['sex', 'pclass'])['survived'].mean().unstack() * 100
tabla_cruzada = tabla_cruzada.round(1)
tabla_cruzada.columns.name = 'Classe'
print("\nTasso di sopravvivenza (%) per sesso e classe:")
print(tabla_cruzada)

# Età × Classe (statistiche)
print("\nEtà media per classe:")
print(df.groupby('pclass')['age'].agg(['mean','median','std']).round(2))

# Famiglia a bordo
df['familia'] = df['sibsp'] + df['parch']
df['tam_familia'] = pd.cut(df['familia'], bins=[-1, 0, 3, 10],
                           labels=['Solo', 'Famiglia piccola', 'Famiglia grande'])
print("\nSopravvivenza per dimensione della famiglia:")
print((df.groupby('tam_familia')['survived'].mean() * 100).round(1))

# Test statistici
print("\n=== Test statistici ===")

# Chi-quadrato: sesso vs sopravvivenza
tabla_chi = pd.crosstab(df['sex'], df['survived'])
chi2, p_chi, _, _ = stats.chi2_contingency(tabla_chi)
print(f"Chi² sesso vs sopravvivenza: χ²={chi2:.2f}, p={p_chi:.4f} → {'Significativo' if p_chi<0.05 else 'Non significativo'}")

# t-test: età dei sopravvissuti vs non sopravvissuti
edad_vivos   = df[df['survived']==1]['age']
edad_muertos = df[df['survived']==0]['age']
t, p_t = stats.ttest_ind(edad_vivos, edad_muertos)
print(f"t-test età: vivi={edad_vivos.mean():.2f}a vs morti={edad_muertos.mean():.2f}a, p={p_t:.4f}")

# t-test: tariffa
fare_vivos   = df[df['survived']==1]['fare']
fare_muertos = df[df['survived']==0]['fare']
t2, p_t2 = stats.ttest_ind(fare_vivos, fare_muertos)
print(f"t-test tariffa: vivi={fare_vivos.mean():.2f}£ vs morti={fare_muertos.mean():.2f}£, p={p_t2:.4f}")

In [ ]:
# Pannello di analisi approfondita
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Heatmap sesso × classe
tabla_heat = df.groupby(['sex', 'pclass'])['survived'].mean().unstack()
im = axes[0,0].imshow(tabla_heat.values, cmap='RdYlGn', vmin=0, vmax=1)
axes[0,0].set_xticks([0,1,2]); axes[0,0].set_yticks([0,1])
axes[0,0].set_xticklabels(['1ª classe', '2ª classe', '3ª classe'])
axes[0,0].set_yticklabels(tabla_heat.index)
for i in range(2):
    for j in range(3):
        axes[0,0].text(j, i, f"{tabla_heat.values[i,j]:.0%}", ha='center', va='center', fontsize=12)
plt.colorbar(im, ax=axes[0,0])
axes[0,0].set_title('Sopravvivenza: Sesso × Classe')

# Boxplot età per classe e sopravvivenza
datos_box = []
labels_box = []
for clase in [1, 2, 3]:
    datos_box.append(df[(df['pclass']==clase)&(df['survived']==0)]['age'].dropna())
    datos_box.append(df[(df['pclass']==clase)&(df['survived']==1)]['age'].dropna())
    labels_box += [f'{clase}ª\nMorto', f'{clase}ª\nVivo']
bplot = axes[0,1].boxplot(datos_box, labels=labels_box, patch_artist=True)
colores = ['#e74c3c','#2ecc71'] * 3
for patch, color in zip(bplot['boxes'], colores):
    patch.set_facecolor(color)
axes[0,1].set_title('Età per classe e sopravvivenza')

# Tariffa per sopravvivenza
axes[0,2].hist(df[df['survived']==0]['fare'].clip(0, 200), bins=30, alpha=0.6,
               color='#e74c3c', label='Non sopravvissuto', density=True)
axes[0,2].hist(df[df['survived']==1]['fare'].clip(0, 200), bins=30, alpha=0.6,
               color='#2ecc71', label='Sopravvissuto', density=True)
axes[0,2].set_title('Distribuzione tariffa (clip 200£)')
axes[0,2].set_xlabel('Tariffa (£)'); axes[0,2].legend()

# Sopravvivenza per gruppo di età
sup_edad = df.groupby('grupo_edad')['survived'].mean() * 100
sup_edad.plot(kind='bar', ax=axes[1,0], color='steelblue')
axes[1,0].set_title('Sopravvivenza per gruppo di età')
axes[1,0].set_ylabel('Tasso (%)'); axes[1,0].set_xticklabels(sup_edad.index, rotation=30, ha='right')

# Sopravvivenza per dimensione della famiglia
sup_fam = df.groupby('tam_familia')['survived'].mean() * 100
sup_fam.plot(kind='bar', ax=axes[1,1], color='coral')
axes[1,1].set_title('Sopravvivenza per dimensione della famiglia')
axes[1,1].set_ylabel('Tasso (%)'); axes[1,1].set_xticklabels(sup_fam.index, rotation=0)

# Correlazione numerica
cols_num = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare', 'familia']
corr = df[cols_num].corr()
im2 = axes[1,2].imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1,2].set_xticks(range(len(cols_num)))
axes[1,2].set_yticks(range(len(cols_num)))
axes[1,2].set_xticklabels(cols_num, rotation=45, ha='right', fontsize=8)
axes[1,2].set_yticklabels(cols_num, fontsize=8)
for i in range(len(cols_num)):
    for j in range(len(cols_num)):
        axes[1,2].text(j, i, f"{corr.values[i,j]:.2f}", ha='center', va='center', fontsize=7)
plt.colorbar(im2, ax=axes[1,2])
axes[1,2].set_title('Correlazioni numeriche')

plt.suptitle('Analisi Titanic — Pattern di Sopravvivenza', fontsize=13)
plt.tight_layout(); plt.show()

---
## Fase 5 — Conclusioni

In [ ]:
print("=" * 60)
print(" CONCLUSIONI DELL'EDA TITANIC")
print("=" * 60)

print("""
DOMANDA: quali fattori determinano la sopravvivenza?

1. Il SESSO è il fattore più determinante:
   - Donne: 74% di sopravvivenza
   - Uomini: 19% di sopravvivenza
   → Protocollo 'donne e bambini prima' chiaramente visibile
   → χ² altamente significativo (p < 0.0001)

2. CLASSE del biglietto:
   - 1ª classe: 63% sopravvivenza
   - 2ª classe: 47% sopravvivenza
   - 3ª classe: 24% sopravvivenza
   → Le cabine di 3ª erano sul ponte inferiore (più lontane dalle scialuppe)

3. L'ETÀ conta, ma meno del previsto:
   - Bambini (<12 anni): 59% → priorità nell'evacuazione
   - Giovani adulti: tasso minore (molti uomini)
   - Differenza di media statisticamente significativa (p=...)

4. TARIFFA (proxy della classe sociale):
   - Alta correlazione con la sopravvivenza
   - I sopravvissuti pagarono in media ~{:.0f}£ contro {:.0f}£ dei deceduti

5. FAMIGLIA a bordo:
   - Famiglie piccole (1-3): tasso migliore
   - Persone sole o famiglie grandi: tasso peggiore
   → Le famiglie grandi impiegavano più tempo a riunirsi per evacuare

IPOTESI PER LA MODELLAZIONE:
  Variabili più predittive: sex, pclass, age, fare, familia
  Possibili interazioni: sex×pclass, age×pclass
  Dato assente: deck (77% mancante, può essere informativo se imputato)
""".format(
    df[df['survived']==1]['fare'].mean(),
    df[df['survived']==0]['fare'].mean()
))

# Tabella riassuntiva finale
print("RIEPILOGO NUMERICO FINALE:")
resumen_final = {
    'Passeggeri totali':      len(df),
    'Sopravvissuti':          df['survived'].sum(),
    'Tasso sopravvivenza':    f"{df['survived'].mean()*100:.1f}%",
    'Età media totale':       f"{df['age'].mean():.1f} anni",
    'Tariffa media':          f"{df['fare'].mean():.2f}£",
    'Viaggiavano da soli':    f"{df['alone'].sum()} ({df['alone'].mean()*100:.0f}%)",
    'Variabili finali':       len(df.columns),
}
for k, v in resumen_final.items():
    print(f"  {k:25}: {v}")

---
## Bonus — Preparazione per il ML

Convertiamo il dataset pulito in un formato pronto per addestrare un modello.

In [ ]:
# Selezionare e trasformare le feature
df_ml = df[['survived', 'pclass', 'sex', 'age', 'fare', 'embarked', 'alone', 'familia']].copy()

# Codifica delle variabili categoriche
df_ml['sex_num']      = (df_ml['sex'] == 'female').astype(int)
df_ml['embarked_C']   = (df_ml['embarked'] == 'C').astype(int)
df_ml['embarked_Q']   = (df_ml['embarked'] == 'Q').astype(int)
df_ml['alone_num']    = df_ml['alone'].astype(int)
df_ml = df_ml.drop(columns=['sex', 'embarked', 'alone'])

# Normalizzazione Min-Max delle variabili continue
for col in ['age', 'fare', 'familia']:
    mn, mx = df_ml[col].min(), df_ml[col].max()
    df_ml[col + '_norm'] = ((df_ml[col] - mn) / (mx - mn)).round(4)

# Feature finali
features = ['pclass', 'sex_num', 'age_norm', 'fare_norm',
            'embarked_C', 'embarked_Q', 'alone_num', 'familia_norm']
X = df_ml[features]
y = df_ml['survived']

print("Dataset pronto per il ML:")
print(f"X shape: {X.shape}")
print(f"y distribuzione: {y.value_counts().to_dict()}")
print("\nPrime righe di X:")
print(X.head())
print("\nCorrelazione feature vs target:")
corr_target = X.corrwith(y).sort_values(key=abs, ascending=False)
print(corr_target.round(4))

print("\nLa feature più correlata con la sopravvivenza: sex_num (donna)")
print("La feature più anticorrelata: pclass (classe più alta = numero minore = maggiore sopravvivenza)")

---
## Pratica

### Esercizio 1 — EDA del dataset `tips`

**Consegna:** Esegui un EDA completo del dataset delle mance del ristorante:

```python
tips = sns.load_dataset('tips')
```

Colonne: `total_bill`, `tip`, `sex`, `smoker`, `day`, `time`, `size`

1. Esplorazione iniziale (shape, tipi, mancanti, describe)
2. Che % di mancia lascia ogni gruppo? (sex, smoker, day, time)
3. Crea la colonna `pct_propina = tip / total_bill * 100`
4. C'è una differenza significativa nella mancia tra fumatori e non fumatori?
5. Visualizza: distribuzione della mancia, mancia vs total_bill, mancia per giorno

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

tips = sns.load_dataset('tips')

# 1. Esplorazione
print(f"Shape: {tips.shape}")
print(tips.dtypes)
print(tips.isnull().sum())
print(tips.describe().round(2))

# 2. Mancia per gruppo
tips['pct_propina'] = (tips['tip'] / tips['total_bill'] * 100).round(2)
for var in ['sex', 'smoker', 'day', 'time']:
    print(f"\nMancia % per {var}:")
    print(tips.groupby(var)['pct_propina'].mean().round(2))

# 4. Test statistico
fumadores     = tips[tips['smoker']=='Yes']['pct_propina']
no_fumadores  = tips[tips['smoker']=='No']['pct_propina']
t, p = stats.ttest_ind(fumadores, no_fumadores)
print(f"\nt-test fumatori vs non fumatori:")
print(f"  fumatori: {fumadores.mean():.2f}%  non_fumatori: {no_fumadores.mean():.2f}%")
print(f"  t={t:.4f}, p={p:.4f} → {'Significativo' if p<0.05 else 'Non significativo'}")

# 5. Visualizzazione
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

tips['pct_propina'].hist(bins=30, ax=axes[0], color='steelblue')
axes[0].set_title('Distribuzione % mancia')
axes[0].set_xlabel('% mancia')

axes[1].scatter(tips['total_bill'], tips['tip'], alpha=0.5, color='coral')
m, b = np.polyfit(tips['total_bill'], tips['tip'], 1)
x = np.linspace(tips['total_bill'].min(), tips['total_bill'].max(), 100)
axes[1].plot(x, m*x+b, 'r-', lw=2)
r, _ = stats.pearsonr(tips['total_bill'], tips['tip'])
axes[1].set_title(f'Mancia vs Totale (r={r:.3f})')
axes[1].set_xlabel('Conto totale ($)'); axes[1].set_ylabel('Mancia ($)')

tips.boxplot(column='pct_propina', by='day', ax=axes[2],
             flierprops=dict(marker='o', ms=3, alpha=0.5))
axes[2].set_title('% Mancia per giorno')
plt.suptitle('')
plt.tight_layout(); plt.show()
```
</details>

---
### Esercizio 2 — Analisi delle correlazioni

**Consegna:** Sul dataset Titanic pulito (`df`) del progetto:

1. Calcola la correlazione di tutte le variabili numeriche con `survived`
2. Crea una nuova feature: `fare_per_person = fare / (familia + 1)`
3. Migliora la correlazione di `fare_per_person` rispetto a `fare` con la sopravvivenza?
4. Visualizza la distribuzione di `fare_per_person` per classe e sopravvivenza
5. Interpreta: questa nuova variabile ha senso dal punto di vista economico?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import matplotlib.pyplot as plt
from scipy import stats

# 1. Correlazioni con survived
cols_num = df.select_dtypes(include='number').columns
corr_surv = df[cols_num].corrwith(df['survived']).sort_values(key=abs, ascending=False)
print("Correlazione con survived:")
print(corr_surv.round(4))

# 2. Nuova feature
df['fare_per_person'] = df['fare'] / (df['familia'] + 1)

# 3. Confrontare la correlazione
r_fare, _  = stats.pearsonr(df['fare'], df['survived'])
r_fpp, _   = stats.pearsonr(df['fare_per_person'], df['survived'])
print(f"\nCorrelazione fare vs survived:           {r_fare:.4f}")
print(f"Correlazione fare_per_person vs survived: {r_fpp:.4f}")
print(f"Migliora? {'Sì' if abs(r_fpp) > abs(r_fare) else 'No'}")

# 4. Visualizzazione
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, var, titulo in zip(axes,
    ['fare', 'fare_per_person'],
    ['Tariffa totale', 'Tariffa per persona']):

    for clase in [1, 2, 3]:
        sub = df[df['pclass'] == clase]
        ax.scatter(sub[var].clip(0, 200), sub['survived'] + (clase-2)*0.1,
                   alpha=0.3, label=f'{clase}ª classe', s=15)
    ax.set_title(titulo)
    ax.set_xlabel('£'); ax.set_ylabel('Sopravvissuto')
    ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

# 5. Interpretazione
print("\nInterpretazione:")
print("  fare_per_person normalizza il prezzo in base alla dimensione della famiglia.")
print("  Un biglietto da £100 per 5 persone (£20/pers) ≠ £100 per 1 (£100/pers).")
print("  Questa variabile può catturare meglio la ricchezza individuale rispetto alla tariffa totale.")
```
</details>

---
### Esercizio 3 — Profilazione dei gruppi

**Consegna:** Individua i 4 gruppi con maggiore e minore sopravvivenza nel Titanic e crea un report:

1. Raggruppa per `sex`, `pclass`, `alone` e calcola il tasso di sopravvivenza + n
2. Filtra i gruppi con almeno 10 persone
3. Mostra il Top 4 (maggiore sopravvivenza) e il Bottom 4 (minore sopravvivenza)
4. Per ogni estremo, mostra anche: età media, tariffa media
5. Visualizza con un grafico a barre orizzontali, colorando in base al tasso

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# 1 e 2. Raggruppamento e filtro
perfil = df.groupby(['sex', 'pclass', 'alone']).agg(
    tasa=('survived', 'mean'),
    n=('survived', 'count'),
    edad_media=('age', 'mean'),
    tarifa_media=('fare', 'mean')
).round(3).reset_index()

perfil = perfil[perfil['n'] >= 10].sort_values('tasa', ascending=False)

# Etichetta leggibile
perfil['grupo'] = perfil.apply(
    lambda r: f"{r['sex']}, {r['pclass']}ª classe, {'solo' if r['alone'] else 'con famiglia'}",
    axis=1
)

# 3 e 4. Top e Bottom
top4    = perfil.head(4)
bottom4 = perfil.tail(4)

for titulo, df_g in [('TOP 4 (maggiore sopravvivenza)', top4),
                      ('BOTTOM 4 (minore sopravvivenza)', bottom4)]:
    print(f"\n{titulo}")
    for _, row in df_g.iterrows():
        print(f"  {row['grupo']:45} tasso={row['tasa']:.0%}  n={row['n']:3.0f}  "
              f"età_media={row['edad_media']:.1f}a  tariffa={row['tarifa_media']:.2f}£")

# 5. Visualizzazione
fig, ax = plt.subplots(figsize=(12, 5))
todos   = pd.concat([top4, bottom4]).sort_values('tasa')
colores = ['#e74c3c' if t < 0.5 else '#2ecc71' for t in todos['tasa']]

bars = ax.barh(todos['grupo'], todos['tasa'] * 100, color=colores)
ax.axvline(50, color='gray', ls='--', lw=1)
for bar, (_, row) in zip(bars, todos.iterrows()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f"n={row['n']:.0f}", va='center', fontsize=9)

ax.set_xlabel('Tasso di sopravvivenza (%)')
ax.set_title('Profili estremi di sopravvivenza sul Titanic')
ax.set_xlim(0, 115)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout(); plt.show()
```
</details>

---
### Esercizio 4 — EDA di un dataset di IA

**Consegna:** Genera e analizza un dataset di un sistema di raccomandazione:

```python
np.random.seed(42)
n_usuarios = 500
eda_df = pd.DataFrame({
    'usuario_id':   range(1, n_usuarios+1),
    'edad':         np.random.normal(35, 12, n_usuarios).clip(18, 70).round(),
    'sesiones':     np.random.poisson(20, n_usuarios),
    'clics':        np.random.poisson(150, n_usuarios),
    'tiempo_min':   np.random.exponential(45, n_usuarios).round(1),
    'compras':      np.random.poisson(3, n_usuarios),
    'segmento':     np.random.choice(['casual','regular','premium'], n_usuarios, p=[0.5,0.35,0.15]),
    'dispositivo':  np.random.choice(['mobile','desktop','tablet'], n_usuarios, p=[0.6,0.3,0.1]),
})
eda_df['tasa_conversion'] = (eda_df['compras'] / eda_df['sesiones']).fillna(0).round(4)
```

Esegui un EDA completo e rispondi: quale segmento è più prezioso per il business?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
n = 500
eda_df = pd.DataFrame({
    'usuario_id':   range(1, n+1),
    'edad':         np.random.normal(35, 12, n).clip(18, 70).round(),
    'sesiones':     np.random.poisson(20, n),
    'clics':        np.random.poisson(150, n),
    'tiempo_min':   np.random.exponential(45, n).round(1),
    'compras':      np.random.poisson(3, n),
    'segmento':     np.random.choice(['casual','regular','premium'], n, p=[0.5,0.35,0.15]),
    'dispositivo':  np.random.choice(['mobile','desktop','tablet'], n, p=[0.6,0.3,0.1]),
})
eda_df['tasa_conversion'] = (eda_df['compras'] / eda_df['sesiones']).fillna(0).round(4)

# Esplorazione
print(f"Shape: {eda_df.shape}")
print(eda_df.describe().round(2))

# Analisi per segmento
valor_seg = eda_df.groupby('segmento').agg(
    n=('usuario_id','count'),
    sesiones_media=('sesiones','mean'),
    compras_media=('compras','mean'),
    conversion=('tasa_conversion','mean'),
    tiempo_media=('tiempo_min','mean')
).round(3)
print("\nMetriche per segmento:")
print(valor_seg)

# Correlazioni
cols_num = ['sesiones','clics','tiempo_min','compras','tasa_conversion']
print("\nCorrelazioni con le compre:")
print(eda_df[cols_num].corrwith(eda_df['compras']).round(4).sort_values(key=abs, ascending=False))

# Visualizzazione
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

segmentos = ['casual','regular','premium']
eda_df.boxplot(column='compras', by='segmento', ax=axes[0],
               flierprops=dict(marker='o', ms=3))
axes[0].set_title('Acquisti per segmento'); plt.sca(axes[0]); plt.title('Acquisti per segmento')

axes[1].scatter(eda_df['sesiones'], eda_df['compras'], alpha=0.3,
                c=eda_df['segmento'].map({'casual':0,'regular':1,'premium':2}),
                cmap='viridis')
axes[1].set_title('Sessioni vs Acquisti'); axes[1].set_xlabel('Sessioni'); axes[1].set_ylabel('Acquisti')

conv_seg = eda_df.groupby('segmento')['tasa_conversion'].mean().reindex(segmentos)
axes[2].bar(segmentos, conv_seg.values * 100, color=['#95a5a6','#3498db','#2ecc71'])
axes[2].set_title('Tasso di conversione per segmento')
axes[2].set_ylabel('%')

plt.tight_layout(); plt.show()

print("\nConclusione: il segmento 'premium' è il più prezioso:")
print(f"  Tasso di conversione più alto: {eda_df[eda_df['segmento']=='premium']['tasa_conversion'].mean():.3f}")
print(f"  Media acquisti più alta:       {eda_df[eda_df['segmento']=='premium']['compras'].mean():.2f}")
print("  Anche se è il 15% degli utenti, genera il maggior valore per persona.")
```
</details>

---
### Esercizio 5 (sfida) — EDA automatizzato

**Consegna:** Implementa una funzione `eda_automatico(df, target)` che:

1. Genera un report testuale: shape, mancanti, tipi, statistiche
2. Per le variabili numeriche: istogramma + correlazione con il target
3. Per le variabili categoriche: frequenze + tasso medio del target per categoria
4. Individua e mostra gli outlier nelle variabili numeriche
5. Restituisce un dict con le 3 variabili più correlate con il target

Provala con il dataset Titanic (`df`, target=`survived`) e con quello di tips (target=`pct_propina`).

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

def eda_automatico(df, target, max_categorias=10):
    print("=" * 60)
    print(f" EDA AUTOMATICO  |  Target: '{target}'")
    print("=" * 60)

    # 1. Report testuale
    print(f"\nShape: {df.shape}")
    print(f"Duplicati: {df.duplicated().sum()}")
    print("\nValori mancanti:")
    miss = df.isnull().sum()
    print(miss[miss > 0].to_string() if miss.sum() > 0 else "  Nessuno")

    cols_num  = df.select_dtypes(include='number').columns.tolist()
    cols_cat  = df.select_dtypes(include=['object','category','bool']).columns.tolist()
    if target in cols_num:  cols_num.remove(target)
    if target in cols_cat:  cols_cat.remove(target)

    print(f"\nVariabili numeriche ({len(cols_num)}): {cols_num}")
    print(f"Variabili categoriche ({len(cols_cat)}): {cols_cat}")

    # 2. Numeriche
    if cols_num:
        n_plots = len(cols_num)
        fig, axes = plt.subplots(1, n_plots, figsize=(min(n_plots*3, 18), 3))
        if n_plots == 1: axes = [axes]
        corr_num = {}
        for ax, col in zip(axes, cols_num):
            datos = df[col].dropna()
            ax.hist(datos, bins=25, density=True, alpha=0.7, color='steelblue')
            r, p = stats.pearsonr(df[col].dropna(), df.loc[df[col].notna(), target])
            corr_num[col] = round(r, 4)
            ax.set_title(f"{col}\nr={r:.3f}", fontsize=9)
        plt.suptitle("Distribuzione variabili numeriche", fontsize=10)
        plt.tight_layout(); plt.show()

    # 3. Categoriche
    if cols_cat:
        n_plots = len(cols_cat)
        fig, axes = plt.subplots(1, n_plots, figsize=(min(n_plots*4, 18), 3))
        if n_plots == 1: axes = [axes]
        for ax, col in zip(axes, cols_cat):
            tasa = df.groupby(col)[target].mean().sort_values(ascending=False).head(max_categorias)
            tasa.plot(kind='bar', ax=ax, color='coral')
            ax.set_title(f"{col} vs {target}", fontsize=9)
            ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
        plt.suptitle("Media del target per categoria", fontsize=10)
        plt.tight_layout(); plt.show()

    # 4. Outlier
    print("\nOutlier (metodo IQR):")
    for col in cols_num:
        Q1, Q3 = df[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        n_out = ((df[col] < Q1-1.5*IQR) | (df[col] > Q3+1.5*IQR)).sum()
        print(f"  {col:20}: {n_out} outlier ({n_out/len(df)*100:.1f}%)")

    # 5. Top correlazioni
    if cols_num:
        top3 = sorted(corr_num.items(), key=lambda x: abs(x[1]), reverse=True)[:3]
        print("\nTop 3 variabili numeriche più correlate con il target:")
        for col, r in top3:
            print(f"  {col}: r={r}")
        return dict(top3)
    return {}

# Prova con Titanic
import seaborn as sns
df_t = sns.load_dataset('titanic').copy()
df_t['age'] = df_t.groupby(['pclass','sex'])['age'].transform(lambda x: x.fillna(x.median()))
df_t['embarked'].fillna('S', inplace=True)
resultado = eda_automatico(df_t[['survived','pclass','age','fare','sex','embarked','alone']], 'survived')
```
</details>

---
## Riepilogo del Modulo 6

```
┌─────────────────────────────────────────────────────────┐
│            MODULO 6 — MATEMATICA PER L'IA                │
├──────────┬──────────────────────────────────────────────┤
│ Lezione 1│ Vettori — NumPy, norma, similarità coseno     │
│ Lezione 2│ Matrici — operazioni, trasformazioni, PCA     │
│ Lezione 3│ Probabilità — Bayes, distribuzioni, Monte C.  │
│ Lezione 4│ Statistica descrittiva — misure, outlier      │
│ Lezione 5│ Inferenza — TLC, intervalli, t-test           │
│ Lezione 6│ pandas — Series, DataFrame, pulizia, merge    │
│ Lezione 7│ EDA completo — Titanic, 5 fasi, preparare ML  │
└──────────┴──────────────────────────────────────────────┘

Pipeline completa che ora sai fare:

  Dati grezzi
      ↓ pandas: caricare, esplorare
  Dati compresi
      ↓ statistica descrittiva: riassumere, visualizzare
  Dati analizzati
      ↓ pulizia: NaN, outlier, tipi
  Dati puliti
      ↓ inferenza: differenze significative?
  Ipotesi validate
      ↓ feature engineering: normalizzare, codificare
  Dataset pronto per il ML → Modulo 7
```

## Cosa vediamo nel prossimo modulo?

**Modulo 7 — Machine Learning**:
Regressione lineare, classificazione, alberi decisionali, valutazione dei modelli e cross-validation. Useremo direttamente i dataset che abbiamo preparato in questo modulo.